# 08A 建立可回查的资料地图

前七节已有模型、工具、检索、上下文、方法、持续任务和评估。本节把它们用于一个具体目标：帮刚入门的学生读懂一份资料和一个小仓库。

先约定结果：学生知道读哪里，能完成一个小动作，并能把关键结论回查到来源。仅仅把整本 PDF 总结成一段长文字，还不够支持动手学习。

**综合案例第一阶段的边界**：08A 建立资料身份和可回查坐标；08B 产出 Mini Guide 学习路径并核验引用；08C 换到固定版本 Hello-Agents 检查迁移。第八节只交付“有证据的导读基线”，练习反馈与学习增量留到第九节。

**贯穿案例**

给学习助手导入包含 PDF 和代码的资料，不只问“搜到了什么”，还要能回到原文。`Corpus.search` 返回候选及资料块编号，`Corpus.read` 按编号取回物理页、提取文本行或源码行；修改资料后旧编号被拒绝，防止过期引用指向新内容。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：开发验收：每个命中都能读回来源。

课件页：L08.02-S01、L08.02-S02、L08.02-S03。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, hashlib, tempfile, copy, subprocess, inspect
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
PROJECT = ROOT / "08 综合实践一 资料与仓库导读/代码/学习助手"
sys.path.insert(0,str(PROJECT / "src"))
from corpus import Corpus
from study_runtime import StudyRuntime, Harness, citation_check
from checkpoint_store import CheckpointStore
from IPython.display import display, Code
print("自编资料默认离线可用；导入不执行仓库代码。")


自编资料默认离线可用；导入不执行仓库代码。


## 1. 一次只完成一小段学习

本次目标是理解 Mini Guide 的检索分数和排序，前提是学生会 Python 函数。完成条件：一条不超过三步的路径、每步有位置和动作、关键解释有来源、最后有一个自检问题。

资料导入只读明确目录。PDF 是自编的两页资料；源码也由课程编写，避免一开始就面对大项目。


In [2]:
task = {"goal":"理解检索分数和排序", "prior_knowledge":["Python 函数"],
    "deliverables":["最多三步的学习路径","每步有位置、动作和自检","有来源的关键解释"]}
print(task)
corpus = Corpus(PROJECT / "fixtures/mini-guide")
print("资料指纹:", corpus.fingerprint)


{'goal': '理解检索分数和排序', 'prior_knowledge': ['Python 函数'], 'deliverables': ['最多三步的学习路径', '每步有位置、动作和自检', '有来源的关键解释']}
资料指纹: be89758248b7b587399212ab18d58e90b83d1b54197a86271c60e9015ad30acc


## 2. 文件地图与知识地图不同

文件地图列出文档、源码、PDF 页数和版本，帮助确定有哪些资料。它没有自动理解所有概念依赖，也不等于适合学生的学习路径。

下面查看实际导入代码，找出目录范围、文件类型、数量/大小、PDF 页码和空页处理。它不调用 import 来执行资料里的 Python 文件。


In [3]:
display(Code(inspect.getsource(Corpus.__init__),language="python"))
display(Code(inspect.getsource(Corpus._read),language="python"))


def __init__(self,directory):
        original=Path(directory)
        if original.is_symlink():raise ValueError('不导入符号链接根目录')
        self.root=original.resolve()
        if not self.root.is_dir():raise ValueError('资料根目录不存在')
        self.documents=[];self.chunks=[];self.locations={};self.hashes={};self.full_documents={}
        paths=[]
        for parent,dirs,files in os.walk(self.root,followlinks=False):
            dirs[:]=sorted(d for d in dirs if not d.startswith('.') and d not in {'node_modules','__pycache__','venv','mock','mocks'}
                           and not (Path(parent)/d).is_symlink())
            for name in sorted(files):
                path=Path(parent)/name
                if name.startswith('.') or '.mock.' in name or path.is_symlink():continue
                if path.suffix.lower() not in {'.md','.txt','.py','.json','.pdf'}:continue
                if len(paths)>=200:raise ValueError('教学资料最多 200 个文件，请先缩小选择')
                paths.append(path)
        if sum(p.stat().st_size for p in paths)>20_000_000:raise ValueError('总资料上限 20 MB')
        for path in paths:self._read(path)
        if not self.chunks:raise ValueError('没有可检索文本')
        self.index=TfidfIndex(self.chunks);self.by_id={c.id:c for c in self.chunks}
        self.by_id.update({c.id:c for c in self.full_documents.values()})
        self.fingerprint=hashlib.sha256(json.dumps(self.hashes,sort_keys=True).encode()).hexdigest()

def _read(self,path):
        relative=path.relative_to(self.root).as_posix();size=path.stat().st_size
        if size>10_000_000:raise ValueError('单文件上限 10 MB')
        digest=hashlib.sha256(path.read_bytes()).hexdigest();self.hashes[relative]=digest
        if path.suffix.lower()=='.pdf':
            reader=PdfReader(path)
            if reader.is_encrypted:raise ValueError('课程导入不处理加密 PDF')
            if not 1<=len(reader.pages)<=80:raise ValueError('教学 PDF 最多 80 页，请先明确选页')
            pages=[(i+1,page.extract_text() or '') for i,page in enumerate(reader.pages)]
        else:
            if size>1_000_000:raise ValueError('文本单文件上限 1 MB')
            pages=[(None,path.read_text(encoding='utf-8'))]
        document={'path':relative,'sha256':digest,'kind':'pdf' if pages[0][0] else 'text',
                  'pages':len(pages),'headings':[],'empty_pages':[]}
        for page,text in pages:
            if len(text)>300_000:raise ValueError('单页/文本提取量过大')
            if not text.strip():document['empty_pages'].append(page);continue
            if page is None:
                document['headings'] += [line.strip('# ')[:100] for line in text.splitlines() if line.startswith('#')][:8]
            source=relative if page is None else f'{relative}#page={page}'
            whole=Chunk(source,1,len(text.splitlines()),text.rstrip('\n'),hashlib.sha256(text.encode()).hexdigest())
            self.full_documents[(relative,page or 0)]=whole
            self.locations[whole.id]={'path':relative,'page':page,'start':1,'end':whole.end_line,
                'line_unit':'source_line' if page is None else 'extracted_text_line','document_sha256':digest}
            for chunk in chunk_text(text,source,lines_per_chunk=12,overlap=2):
                self.chunks.append(chunk)
                self.locations[chunk.id]={'path':relative,'page':page,'start':chunk.start_line,'end':chunk.end_line,
                    'line_unit':'source_line' if page is None else 'extracted_text_line','document_sha256':digest}
        self.documents.append(document)

In [4]:
for document in corpus.map()["documents"]:
    print(document["path"],document["kind"],"页/文件数:",document["pages"],"空文本页:",document["empty_pages"])
print("资料块数:",len(corpus.chunks))
# headings 只是 Markdown 标题，不是自动推断出的知识先修图。


README.md text 页/文件数: 1 空文本页: []
learning-notes.pdf pdf 页/文件数: 2 空文本页: []
main.py text 页/文件数: 1 空文本页: []
search.py text 页/文件数: 1 空文本页: []
资料块数: 7


## 3. PDF 物理页与提取文本行

PDF 第一页指文件中的第一个物理页面，不一定等于印在书上的页码。提取后的第几行由解析器输出决定，可能包含页脚或采用不同的阅读顺序；不能冒充原稿印刷行号。

本实现保留页码、原文件 SHA-256、提取文本行范围和文本哈希。扫描页可能没有文本，此时应报告需 OCR，而不是声称资料为空。


In [5]:
from pypdf import PdfReader
pdf_path = PROJECT / "fixtures/mini-guide/learning-notes.pdf"
reader = PdfReader(pdf_path)
print("物理页数:",len(reader.pages))
text = reader.pages[0].extract_text()
for i,line in enumerate(text.splitlines()[:12],1):print(i,line)
assert len(reader.pages)==2
# 打开同目录 PDF，对照为什么页脚可能出现在提取顺序的前面。


物理页数: 2
1 Hands-on LLM Apps / 自编教学资料
2 1
3 带着证据学习资料
4 01 / 资料、检索与解释
5 资料块是可定位的一段原文。代码通常按源文件行号定位；PDF
6 则先保留文件物理页码，再记录该页提取文本中的位置。
7 检索把问题与资料作比较，返回相关片段。Mini Guide 的
8 search.py 按出现的查询词数量打分，它没有训练语义编码器。
9 分数高只说明在这个检索方法下更相关，不代表事实正确的概率。
10 资料本身可能过时，片段也可能缺少前提。
11 学习顺序：先读 README，找到函数入口；再读 search.py，
12 预测输出；最后改变 top_k，解释结果条数与排序。


## 4. 检索后读回完整原文

复用第三节 TF-IDF：中文二字片段与英文标识符是检索单元，不是模型 Token。这里先用关键词把过程做清楚，语义编码仍可按第三节方式替换。

检索返回短预览；用于解释时再读取完整块。ID 绑定版本和位置，避免资料更新后拿旧位置冒充新证据。


In [6]:
display(Code(inspect.getsource(Corpus.search),language="python"))
display(Code(inspect.getsource(Corpus.read),language="python"))
display(Code(inspect.getsource(Corpus.read_document),language="python"))


def search(self,query,k=3):
        if not isinstance(query,str) or not 1<=len(query.strip())<=300:raise ValueError('查询需为 1–300 字符')
        if type(k) is not int or not 1<=k<=5:raise ValueError('k 为 1–5')
        return {'hits':[{'id':h['chunk'].id,'score':round(h['score'],6),'location':self.locations[h['chunk'].id],
                        'preview':h['chunk'].text[:400]} for h in self.index.search(query,k)],
                'score_meaning':'TF-IDF 余弦相似度，不是正确概率'}

def read(self,chunk_id):
        if not isinstance(chunk_id,str) or chunk_id not in self.by_id:raise ValueError('资料块 ID 不存在')
        c=self.by_id[chunk_id]
        if len(c.text)>6000:raise ValueError('此块超过读取上限，需要更细的资料预处理')
        return {'id':c.id,'location':self.locations[c.id],'text':c.text,'text_sha256':c.source_sha256}

def read_document(self,path,page=0):
        """短文本全文或 PDF 单页；长文本仍需先检索分块。page=0 表示普通文本。"""
        if not isinstance(path,str) or type(page) is not int or (path,page) not in self.full_documents:
            raise ValueError('文件或物理页不存在；普通文本 page=0')
        return self.read(self.full_documents[(path,page)].id)

In [7]:
hits = corpus.search("检索 分数 概率",k=3)["hits"]
for hit in hits:print(hit["id"],hit["score"],hit["location"])
pdf_hit = next(h for h in hits if h["location"]["page"]==1)
evidence = corpus.read(pdf_hit["id"])
print(evidence["text"])
assert evidence["location"]["line_unit"]=="extracted_text_line"


c3eea293f7beb71f 0.166128 {'path': 'learning-notes.pdf', 'page': 1, 'start': 1, 'end': 12, 'line_unit': 'extracted_text_line', 'document_sha256': '1bd45934821031a5fe0b3911a3217ee0624221d6629b7d118adcf438b52202a8'}
47d1166c1fe3a774 0.13875 {'path': 'README.md', 'page': None, 'start': 1, 'end': 12, 'line_unit': 'source_line', 'document_sha256': '7260cdc9bad8408fd8a4599eb4d09adca1bddaa47df09b6326385fc65b3471bc'}
0f34f654c0736213 0.030877 {'path': 'learning-notes.pdf', 'page': 2, 'start': 1, 'end': 12, 'line_unit': 'extracted_text_line', 'document_sha256': '1bd45934821031a5fe0b3911a3217ee0624221d6629b7d118adcf438b52202a8'}
Hands-on LLM Apps / 自编教学资料
1
带着证据学习资料
01 / 资料、检索与解释
资料块是可定位的一段原文。代码通常按源文件行号定位；PDF
则先保留文件物理页码，再记录该页提取文本中的位置。
检索把问题与资料作比较，返回相关片段。Mini Guide 的
search.py 按出现的查询词数量打分，它没有训练语义编码器。
分数高只说明在这个检索方法下更相关，不代表事实正确的概率。
资料本身可能过时，片段也可能缺少前提。
学习顺序：先读 README，找到函数入口；再读 search.py，
预测输出；最后改变 top_k，解释结果条数与排序。


### 开发验收：每个命中都能读回来源

配套课件：L08.02-S01、L08.02-S02、L08.02-S03。

导入器输出片段和位置；搜索器只选候选；读取接口负责按 ID 返回对应正文。开发者应对每个命中执行一次读回，检查位置一致，并对照 PDF 物理页，而不是把搜索摘要当成完整原文。

In [8]:
for case_hit in hits:
    case_read = corpus.read(case_hit["id"])
    case_loc = case_read["location"]
    print("命中:", case_hit["id"], "文件:", case_loc["path"], "物理页:", case_loc["page"],
          "行:", case_loc["start"], "至", case_loc["end"], "单位:", case_loc["line_unit"])
    assert case_read["location"] == case_hit["location"]
    assert case_read["text"].strip()
print("PDF 第 1 个物理页字符数:", len(reader.pages[0].extract_text() or ""))
print("导入版本:", corpus.fingerprint)


命中: c3eea293f7beb71f 文件: learning-notes.pdf 物理页: 1 行: 1 至 12 单位: extracted_text_line
命中: 47d1166c1fe3a774 文件: README.md 物理页: None 行: 1 至 12 单位: source_line
命中: 0f34f654c0736213 文件: learning-notes.pdf 物理页: 2 行: 1 至 12 单位: extracted_text_line
PDF 第 1 个物理页字符数: 339
导入版本: be89758248b7b587399212ab18d58e90b83d1b54197a86271c60e9015ad30acc


**结果解读**

能读回且位置一致是接口级验收；仍需肉眼对照 PDF，检查提取顺序和表格错位。PDF 页码是物理页，extracted_text_line 是提取文本的行号，不等同于原页面的视觉行。

**小练习**

给导入器补一个“扫描 PDF 提取为空”的验收条件，应返回什么状态，为什么不能装作已读懂？

<details><summary>完成后展开参考分析</summary>

应记录空文本页并明确需要 OCR 或人工处理；空字符串不能当作不存在相关知识，更不能让模型补写原文。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 练习：更新原文以后，旧 ID 还能用吗

在临时目录写两行说明，取得块 ID；修改第二行后重新导入，再尝试读取旧 ID。先预测结果，并说明为什么不能只保留文件名与行号。


In [9]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "notes.md"
    path.write_text("# 说明\n旧内容", encoding='utf-8')
    before = Corpus(directory); old_id = before.chunks[0].id
    path.write_text("# 说明\n已修订内容", encoding='utf-8')
    after = Corpus(directory)
    print("版本是否变化:",before.fingerprint!=after.fingerprint)
    try:after.read(old_id)
    except ValueError as error:print("预期拒绝:",error)


版本是否变化: True
预期拒绝: 资料块 ID 不存在


In [10]:
assert before.fingerprint!=after.fingerprint
assert old_id not in after.by_id
print("引用需要资料版本；旧内容不能无提示地对应到新位置。")


引用需要资料版本；旧内容不能无提示地对应到新位置。


## 交付：资料地图审计记录

将资料身份、文件与 PDF 坐标、本次检索命中和旧 ID 拒绝结果落盘。这份记录是后续学习路径的证据入口，不是另一份资料摘要。


In [11]:
delivery_dir = ROOT / ".local/student-runs/l08"
delivery_dir.mkdir(parents=True, exist_ok=True)
source_audit = {
    "artifact_type": "source_map_audit",
    "corpus_fingerprint": corpus.fingerprint,
    "documents": corpus.map()["documents"],
    "retrieval_query": "检索 分数 概率",
    "retrieval_ids": [hit["id"] for hit in hits],
    "retrieval_coordinates": [hit["location"] for hit in hits],
    "stale_id_rejected": old_id not in after.by_id,
    "coordinate_note": "PDF 使用物理页与提取文本行；代码使用文件路径与行范围。",
}
(delivery_dir / "source-map-audit.json").write_text(
    json.dumps(source_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
(delivery_dir / "source-map-audit.md").write_text(
    "# 资料地图审计\n\n"
    f"- 资料指纹：`{source_audit['corpus_fingerprint']}`\n"
    f"- 检索命中：{len(source_audit['retrieval_ids'])} 个\n"
    f"- 旧 ID 已拒绝：{source_audit['stale_id_rejected']}\n"
    f"- 坐标说明：{source_audit['coordinate_note']}\n",
    encoding="utf-8",
)
print("已保存:", delivery_dir / "source-map-audit.json")
print("旧 ID 拒绝:", source_audit["stale_id_rejected"])


已保存: .local\student-runs\l08\source-map-audit.json
旧 ID 拒绝: True


## 回顾

明确一小段学习目标；导入只读资料；区分文件地图与学习路径；PDF 保留物理页和提取行；检索后读原文；引用绑定版本。

下一份实验让同一 Harness 组织这些工具。参考：[pypdf 文本提取说明](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)。
